In [0]:
from pyspark.sql.functions import year, count, countDistinct, when, col

# Read the subscription fact table
df = spark.table("gold_catalog.default.fact_subscription_table")

# Calculate customers gained and lost by year
result = df.groupBy(year(col("start_date")).alias("year")).agg(
    countDistinct(when(col("close_status").isin(["WON", "ACTIVE"]), col("customer_id"))).alias("customers_gained"),
    countDistinct(when(col("close_status") == "LOST", col("customer_id"))).alias("customers_lost")
).withColumn(
    "net_change", 
    col("customers_gained") - col("customers_lost")
).orderBy("year")

display(result)

In [0]:
from pyspark.sql.functions import year, countDistinct, when, col, lag, round as spark_round
from pyspark.sql.window import Window

# Read the subscription fact table
df = spark.table("gold_catalog.default.fact_subscription_table")

# Calculate customers lost by year with trend analysis
loss_trend = df.groupBy(year(col("start_date")).alias("year")).agg(
    countDistinct(when(col("close_status") == "LOST", col("customer_id"))).alias("customers_lost"),
    countDistinct(col("customer_id")).alias("total_customers")
).withColumn(
    "churn_rate_percent",
    spark_round((col("customers_lost") / col("total_customers")) * 100, 2)
).orderBy("year")

# Add period-over-period comparison using window function
window_spec = Window.orderBy("year")
loss_comparison = loss_trend.withColumn(
    "previous_year_lost",
    lag(col("customers_lost")).over(window_spec)
).withColumn(
    "change_vs_previous",
    col("customers_lost") - col("previous_year_lost")
).withColumn(
    "change_percent",
    spark_round(((col("customers_lost") - col("previous_year_lost")) / col("previous_year_lost")) * 100, 2)
)

display(loss_comparison)

In [0]:
from pyspark.sql.functions import sum as _sum, col, round as spark_round, count

# Read fact and dimension tables
fact_df = spark.table("gold_catalog.default.fact_subscription_table")
dim_customer = spark.table("gold_catalog.default.dim_customer")

# Calculate total MRR by customer with customer details
highest_value_customers = fact_df.filter(col("close_status").isin(["ACTIVE", "WON"])) \
    .groupBy("customer_id").agg(
        _sum("MRR_GBP").alias("total_mrr_gbp"),
        _sum("Revenue_GBP").alias("total_revenue_gbp"),
        count("opportunity_id").alias("active_subscriptions")
    ) \
    .join(dim_customer, "customer_id", "inner") \
    .select(
        col("customer_id"),
        col("customer_name"),
        col("industry_type"),
        col("country_name"),
        spark_round(col("total_mrr_gbp"), 2).alias("total_mrr_gbp"),
        spark_round(col("total_revenue_gbp"), 2).alias("total_revenue_gbp"),
        col("active_subscriptions")
    ) \
    .orderBy(col("total_mrr_gbp").desc())

display(highest_value_customers.limit(20))

In [0]:
from pyspark.sql.functions import year, col, sum as _sum, count, countDistinct, round as spark_round, lag
from pyspark.sql.window import Window

# Read fact and dimension tables
fact_df = spark.table("gold_catalog.default.fact_subscription_table")
dim_customer = spark.table("gold_catalog.default.dim_customer")

# Calculate yearly metrics per customer for active subscriptions
customer_yearly_metrics = fact_df.filter(col("close_status").isin(["ACTIVE", "WON"])) \
    .groupBy("customer_id", year(col("start_date")).alias("year")).agg(
        countDistinct("product_id").alias("unique_products"),
        count("opportunity_id").alias("total_subscriptions"),
        _sum("MRR_GBP").alias("total_mrr_gbp"),
        _sum("Revenue_GBP").alias("total_revenue_gbp")
    )

# Use window function to compare year-over-year growth
window_spec = Window.partitionBy("customer_id").orderBy("year")

customer_growth = customer_yearly_metrics.withColumn(
    "prev_year_products",
    lag(col("unique_products")).over(window_spec)
).withColumn(
    "prev_year_mrr",
    lag(col("total_mrr_gbp")).over(window_spec)
).withColumn(
    "product_growth",
    col("unique_products") - col("prev_year_products")
).withColumn(
    "mrr_growth",
    col("total_mrr_gbp") - col("prev_year_mrr")
).withColumn(
    "mrr_growth_percent",
    spark_round(((col("total_mrr_gbp") - col("prev_year_mrr")) / col("prev_year_mrr")) * 100, 2)
)

# Filter customers with positive growth and join with customer details
expanding_customers = customer_growth.filter(
    (col("product_growth") > 0) | (col("mrr_growth_percent") > 10)
).join(dim_customer, "customer_id", "inner") \
    .select(
        col("customer_id"),
        col("customer_name"),
        col("industry_type"),
        col("country_name"),
        col("year"),
        col("unique_products"),
        col("prev_year_products"),
        col("product_growth"),
        spark_round(col("total_mrr_gbp"), 2).alias("total_mrr_gbp"),
        spark_round(col("prev_year_mrr"), 2).alias("prev_year_mrr"),
        spark_round(col("mrr_growth"), 2).alias("mrr_growth"),
        col("mrr_growth_percent")
    ).orderBy(col("mrr_growth").desc())

display(expanding_customers.limit(50))

In [0]:
from pyspark.sql.functions import year, col, sum as _sum, count, countDistinct, round as spark_round, lag, max as _max
from pyspark.sql.window import Window

# Read fact and dimension tables
fact_df = spark.table("gold_catalog.default.fact_subscription_table")
dim_customer = spark.table("gold_catalog.default.dim_customer")

# Calculate yearly metrics per customer
customer_yearly_metrics = fact_df.filter(col("close_status").isin(["ACTIVE", "WON"])) \
    .groupBy("customer_id", year(col("start_date")).alias("year")).agg(
        countDistinct("product_id").alias("unique_products"),
        count("opportunity_id").alias("total_subscriptions"),
        _sum("MRR_GBP").alias("total_mrr_gbp"),
        _sum("Revenue_GBP").alias("total_revenue_gbp")
    )

# Use window function to compare year-over-year changes
window_spec = Window.partitionBy("customer_id").orderBy("year")

customer_decline = customer_yearly_metrics.withColumn(
    "prev_year_products",
    lag(col("unique_products")).over(window_spec)
).withColumn(
    "prev_year_mrr",
    lag(col("total_mrr_gbp")).over(window_spec)
).withColumn(
    "product_change",
    col("unique_products") - col("prev_year_products")
).withColumn(
    "mrr_change",
    col("total_mrr_gbp") - col("prev_year_mrr")
).withColumn(
    "mrr_change_percent",
    spark_round(((col("total_mrr_gbp") - col("prev_year_mrr")) / col("prev_year_mrr")) * 100, 2)
)

# Filter customers with negative growth (reducing usage)
reducing_customers = customer_decline.filter(
    (col("product_change") < 0) | (col("mrr_change_percent") < -10)
).join(dim_customer, "customer_id", "inner") \
    .select(
        col("customer_id"),
        col("customer_name"),
        col("industry_type"),
        col("country_name"),
        col("year"),
        col("unique_products"),
        col("prev_year_products"),
        col("product_change"),
        spark_round(col("total_mrr_gbp"), 2).alias("total_mrr_gbp"),
        spark_round(col("prev_year_mrr"), 2).alias("prev_year_mrr"),
        spark_round(col("mrr_change"), 2).alias("mrr_change"),
        col("mrr_change_percent")
    ).orderBy(col("mrr_change").asc())

# Also identify churned customers (completely stopped)
churned_customers = fact_df.filter(col("close_status") == "LOST") \
    .groupBy("customer_id").agg(
        _max(year(col("end_date"))).alias("last_active_year"),
        _sum("Revenue_GBP").alias("total_lost_revenue"),
        count("opportunity_id").alias("lost_subscriptions")
    ).join(dim_customer, "customer_id", "inner") \
    .select(
        col("customer_id"),
        col("customer_name"),
        col("industry_type"),
        col("country_name"),
        col("last_active_year"),
        spark_round(col("total_lost_revenue"), 2).alias("total_lost_revenue"),
        col("lost_subscriptions")
    ).orderBy(col("total_lost_revenue").desc())

print("=== Customers Reducing Usage ===")
display(reducing_customers.limit(30))

print("\n=== Churned Customers (Completely Stopped) ===")
display(churned_customers.limit(30))

In [0]:
from pyspark.sql.functions import year, col, sum as _sum, countDistinct, round as spark_round, when, min as _min, max as _max

# Read fact table
fact_df = spark.table("gold_catalog.default.fact_subscription_table")

# Get the available years in the data
year_range = fact_df.select(
    _min(year(col("start_date"))).alias("min_year"),
    _max(year(col("start_date"))).alias("max_year")
).collect()[0]

print(f"Data available from {year_range['min_year']} to {year_range['max_year']}")

# Calculate MRR by customer and year for active subscriptions
customer_mrr_by_year = fact_df.filter(col("close_status").isin(["ACTIVE", "WON"])) \
    .groupBy("customer_id", year(col("start_date")).alias("year")).agg(
        _sum("MRR_GBP").alias("mrr_gbp")
    )

# For each year, calculate retention from previous year
retention_results = []

for current_year in range(year_range['min_year'] + 1, year_range['max_year'] + 1):
    previous_year = current_year - 1
    
    # Get customers from previous year
    prev_year_customers = customer_mrr_by_year.filter(col("year") == previous_year) \
        .select(
            col("customer_id").alias("prev_customer_id"),
            col("mrr_gbp").alias("prev_mrr")
        )
    
    # Get customers from current year
    curr_year_customers = customer_mrr_by_year.filter(col("year") == current_year) \
        .select(
            col("customer_id").alias("curr_customer_id"),
            col("mrr_gbp").alias("curr_mrr")
        )
    
    # Join to find existing customers and their MRR changes
    retention_analysis = prev_year_customers.join(
        curr_year_customers,
        prev_year_customers["prev_customer_id"] == curr_year_customers["curr_customer_id"],
        "left"
    )
    
    # Calculate retention metrics
    metrics = retention_analysis.agg(
        _sum("prev_mrr").alias("total_prev_mrr"),
        _sum(when(col("curr_mrr").isNotNull(), col("curr_mrr")).otherwise(0)).alias("total_retained_mrr"),
        _sum(when(col("curr_mrr") >= col("prev_mrr"), col("prev_mrr")).otherwise(col("curr_mrr"))).alias("gross_retained_mrr"),
        countDistinct("prev_customer_id").alias("total_customers"),
        countDistinct(when(col("curr_mrr").isNotNull(), col("prev_customer_id"))).alias("retained_customers")
    ).collect()[0]
    
    # Calculate retention rates
    total_prev_mrr = metrics['total_prev_mrr'] or 0
    total_retained_mrr = metrics['total_retained_mrr'] or 0
    gross_retained_mrr = metrics['gross_retained_mrr'] or 0
    
    nrr = (total_retained_mrr / total_prev_mrr * 100) if total_prev_mrr > 0 else 0
    grr = (gross_retained_mrr / total_prev_mrr * 100) if total_prev_mrr > 0 else 0
    customer_retention = (metrics['retained_customers'] / metrics['total_customers'] * 100) if metrics['total_customers'] > 0 else 0
    
    retention_results.append((
        previous_year,
        current_year,
        round(total_prev_mrr, 2),
        round(total_retained_mrr, 2),
        round(gross_retained_mrr, 2),
        round(nrr, 2),
        round(grr, 2),
        metrics['total_customers'],
        metrics['retained_customers'],
        round(customer_retention, 2)
    ))

# Create DataFrame from results
retention_df = spark.createDataFrame(
    retention_results,
    ["previous_year", "current_year", "previous_year_mrr", "retained_mrr", "gross_retained_mrr",
     "net_revenue_retention_percent", "gross_revenue_retention_percent",
     "total_customers", "retained_customers", "customer_retention_percent"]
)

print("\n=== Revenue Retention Analysis Year-over-Year ===")
print("NRR (Net Revenue Retention): Includes expansion revenue from existing customers")
print("GRR (Gross Revenue Retention): Excludes expansion, only measures what was retained\n")

display(retention_df)

In [0]:
from pyspark.sql.functions import year, col, sum as _sum, round as spark_round, when, max as _max

# Read fact table
fact_df = spark.table("gold_catalog.default.fact_subscription_table")

# Determine the current FY (latest year in data)
current_fy = fact_df.select(_max(year(col("start_date"))).alias("max_year")).collect()[0]['max_year']
previous_fy = current_fy - 1

print(f"Analyzing revenue growth from FY {previous_fy} to FY {current_fy}")
print("=" * 60)

# Get MRR by customer for previous year
prev_year_mrr = fact_df.filter(
    (year(col("start_date")) == previous_fy) & 
    (col("close_status").isin(["ACTIVE", "WON"]))
).groupBy("customer_id").agg(
    _sum("MRR_GBP").alias("prev_mrr")
)

# Get MRR by customer for current year
current_year_mrr = fact_df.filter(
    (year(col("start_date")) == current_fy) & 
    (col("close_status").isin(["ACTIVE", "WON"]))
).groupBy("customer_id").agg(
    _sum("MRR_GBP").alias("curr_mrr")
)

# Full outer join to capture all customers from both periods
all_customers = prev_year_mrr.join(
    current_year_mrr, 
    "customer_id", 
    "full_outer"
).fillna(0, subset=["prev_mrr", "curr_mrr"])

# Categorize each customer's contribution
categorized_customers = all_customers.withColumn(
    "category",
    when((col("prev_mrr") == 0) & (col("curr_mrr") > 0), "new")
    .when((col("prev_mrr") > 0) & (col("curr_mrr") == 0), "churned")
    .when((col("prev_mrr") > 0) & (col("curr_mrr") > col("prev_mrr")), "expansion")
    .when((col("prev_mrr") > 0) & (col("curr_mrr") < col("prev_mrr")), "contraction")
    .otherwise("retained_flat")
).withColumn(
    "mrr_change",
    col("curr_mrr") - col("prev_mrr")
)

# Calculate summary metrics by category
summary = categorized_customers.groupBy("category").agg(
    _sum("prev_mrr").alias("total_prev_mrr"),
    _sum("curr_mrr").alias("total_curr_mrr"),
    _sum("mrr_change").alias("total_mrr_change")
).withColumn(
    "total_prev_mrr", spark_round(col("total_prev_mrr"), 2)
).withColumn(
    "total_curr_mrr", spark_round(col("total_curr_mrr"), 2)
).withColumn(
    "total_mrr_change", spark_round(col("total_mrr_change"), 2)
).orderBy(
    when(col("category") == "new", 1)
    .when(col("category") == "expansion", 2)
    .when(col("category") == "retained_flat", 3)
    .when(col("category") == "contraction", 4)
    .when(col("category") == "churned", 5)
)

print("\n=== MRR Movement Breakdown ===")
display(summary)

# Calculate overall metrics
metrics = categorized_customers.agg(
    _sum("prev_mrr").alias("starting_mrr"),
    _sum(when(col("category") == "new", col("mrr_change")).otherwise(0)).alias("new_mrr"),
    _sum(when(col("category") == "expansion", col("mrr_change")).otherwise(0)).alias("expansion_mrr"),
    _sum(when(col("category") == "contraction", col("mrr_change")).otherwise(0)).alias("contraction_mrr"),
    _sum(when(col("category") == "churned", col("mrr_change")).otherwise(0)).alias("churned_mrr"),
    _sum("curr_mrr").alias("ending_mrr")
).collect()[0]

# Create waterfall breakdown
waterfall_data = [
    (f"FY {previous_fy} Starting MRR", round(metrics['starting_mrr'], 2), "baseline"),
    ("New Customers", round(metrics['new_mrr'], 2), "gain"),
    ("Expansion (Upgrades)", round(metrics['expansion_mrr'], 2), "gain"),
    ("Contraction (Downgrades)", round(metrics['contraction_mrr'], 2), "loss"),
    ("Churned Customers", round(metrics['churned_mrr'], 2), "loss"),
    (f"FY {current_fy} Ending MRR", round(metrics['ending_mrr'], 2), "baseline")
]

waterfall_df = spark.createDataFrame(waterfall_data, ["metric", "value", "type"])

print("\n=== MRR Waterfall Analysis ===")
display(waterfall_df)

# Calculate key percentages
net_growth = metrics['ending_mrr'] - metrics['starting_mrr']
growth_rate = (net_growth / metrics['starting_mrr'] * 100) if metrics['starting_mrr'] > 0 else 0
expansion_contribution = (metrics['expansion_mrr'] / net_growth * 100) if net_growth > 0 else 0

print(f"\n=== Key Insights ===")
print(f"Starting MRR (FY {previous_fy}): £{metrics['starting_mrr']:,.2f}")
print(f"Ending MRR (FY {current_fy}): £{metrics['ending_mrr']:,.2f}")
print(f"Net MRR Growth: £{net_growth:,.2f} ({growth_rate:.2f}%)")
print(f"\nGrowth Sources:")
print(f"  • Expansion (Upgrades): £{metrics['expansion_mrr']:,.2f}")
print(f"  • New Customers: £{metrics['new_mrr']:,.2f}")
print(f"\nRevenue Losses:")
print(f"  • Contraction (Downgrades): £{metrics['contraction_mrr']:,.2f}")
print(f"  • Churned Customers: £{metrics['churned_mrr']:,.2f}")
print(f"\nExpansion contribution to growth: {expansion_contribution:.1f}%")

In [0]:
from pyspark.sql.functions import year, month, col, sum as _sum, round as spark_round, max as _max, dayofyear, when

# Read fact table
fact_df = spark.table("gold_catalog.default.fact_subscription_table")

# Determine the latest year in data
latest_year = fact_df.select(_max(year(col("start_date"))).alias("max_year")).collect()[0]['max_year']
previous_year = latest_year - 1

# Get the latest day of year available in the latest year
latest_day_of_year = fact_df.filter(year(col("start_date")) == latest_year) \
    .select(_max(dayofyear(col("start_date"))).alias("max_doy")).collect()[0]['max_doy']

print(f"Comparing YTD Revenue: {previous_year} vs {latest_year}")
print(f"Period: Day 1 to Day {latest_day_of_year} of each year")
print("="*70)

# Calculate YTD revenue for both years (same day-of-year range)
current_ytd = fact_df.filter(
    (year(col("start_date")) == latest_year) & 
    (dayofyear(col("start_date")) <= latest_day_of_year)
).agg(
    _sum("Revenue_GBP").alias("total_revenue"),
    _sum("MRR_GBP").alias("total_mrr")
).collect()[0]

previous_ytd = fact_df.filter(
    (year(col("start_date")) == previous_year) & 
    (dayofyear(col("start_date")) <= latest_day_of_year)
).agg(
    _sum("Revenue_GBP").alias("total_revenue"),
    _sum("MRR_GBP").alias("total_mrr")
).collect()[0]

# Calculate growth metrics
revenue_growth = current_ytd['total_revenue'] - previous_ytd['total_revenue']
revenue_growth_pct = (revenue_growth / previous_ytd['total_revenue'] * 100) if previous_ytd['total_revenue'] > 0 else 0

mrr_growth = current_ytd['total_mrr'] - previous_ytd['total_mrr']
mrr_growth_pct = (mrr_growth / previous_ytd['total_mrr'] * 100) if previous_ytd['total_mrr'] > 0 else 0

# Create summary dataframe
summary_data = [
    ("Revenue (GBP)", round(previous_ytd['total_revenue'], 2), round(current_ytd['total_revenue'], 2), 
     round(revenue_growth, 2), round(revenue_growth_pct, 2)),
    ("MRR (GBP)", round(previous_ytd['total_mrr'], 2), round(current_ytd['total_mrr'], 2), 
     round(mrr_growth, 2), round(mrr_growth_pct, 2))
]

summary_df = spark.createDataFrame(
    summary_data,
    ["metric", f"ytd_{previous_year}", f"ytd_{latest_year}", "absolute_change", "percent_change"]
)

print("\n=== YTD Revenue Comparison Summary ===")
display(summary_df)

# Monthly breakdown comparison
monthly_comparison = fact_df.filter(
    (year(col("start_date")).isin([previous_year, latest_year])) &
    (dayofyear(col("start_date")) <= latest_day_of_year)
).groupBy(
    year(col("start_date")).alias("year"),
    month(col("start_date")).alias("month")
).agg(
    _sum("Revenue_GBP").alias("monthly_revenue"),
    _sum("MRR_GBP").alias("monthly_mrr")
).orderBy("year", "month")

# Pivot to show side-by-side comparison
monthly_pivot = monthly_comparison.groupBy("month").pivot("year").agg(
    spark_round(_sum("monthly_revenue"), 2).alias("revenue")
).withColumn(
    "month_over_month_change",
    spark_round(col(f"`{latest_year}`") - col(f"`{previous_year}`"), 2)
).withColumn(
    "month_over_month_change_pct",
    spark_round(
        when(col(f"`{previous_year}`") > 0, 
             ((col(f"`{latest_year}`") - col(f"`{previous_year}`")) / col(f"`{previous_year}`")) * 100
        ).otherwise(0), 
        2
    )
).orderBy("month")

print("\n=== Monthly Revenue Comparison ===")
display(monthly_pivot)

# Print key insights
print(f"\n=== Key Insights ===")
print(f"\nRevenue Performance:")
print(f"  • YTD {previous_year}: {previous_ytd['total_revenue']:,.2f}")
print(f"  • YTD {latest_year}: {current_ytd['total_revenue']:,.2f}")
print(f"  • Change: {revenue_growth:,.2f} ({revenue_growth_pct:+.2f}%)")

print(f"\nMRR Performance:")
print(f"  • YTD {previous_year}: {previous_ytd['total_mrr']:,.2f}")
print(f"  • YTD {latest_year}: {current_ytd['total_mrr']:,.2f}")
print(f"  • Change: {mrr_growth:,.2f} ({mrr_growth_pct:+.2f}%)")

if revenue_growth_pct > 0:
    print(f"\n Revenue is UP {revenue_growth_pct:.2f}% year-over-year")
else:
    print(f"\n Revenue is DOWN {abs(revenue_growth_pct):.2f}% year-over-year")

In [0]:
from pyspark.sql.functions import sum as _sum, col, round as spark_round, count, desc, countDistinct
from pyspark.sql.window import Window

# Read fact and dimension tables
fact_df = spark.table("gold_catalog.default.fact_subscription_table")
dim_customer = spark.table("gold_catalog.default.dim_customer")
dim_product = spark.table("gold_catalog.default.dim_product")

# Filter for active subscriptions only
active_subscriptions = fact_df.filter(col("close_status").isin(["ACTIVE", "WON"]))

# Calculate total recurring revenue
total_mrr = active_subscriptions.agg(_sum("MRR_GBP").alias("total")).collect()[0]['total']

print(f"Total Active MRR: £{total_mrr:,.2f}")
print("=" * 80)

# ===== TOP CUSTOMERS BY RECURRING REVENUE =====
top_customers = active_subscriptions.groupBy("customer_id").agg(
    _sum("MRR_GBP").alias("total_mrr_gbp"),
    _sum("Revenue_GBP").alias("total_revenue_gbp"),
    count("opportunity_id").alias("active_subscriptions")
).withColumn(
    "mrr_contribution_percent",
    spark_round((col("total_mrr_gbp") / total_mrr) * 100, 2)
).join(dim_customer, "customer_id", "inner") \
    .select(
        col("customer_id"),
        col("customer_name"),
        col("industry_type"),
        col("country_name"),
        spark_round(col("total_mrr_gbp"), 2).alias("total_mrr_gbp"),
        spark_round(col("total_revenue_gbp"), 2).alias("total_revenue_gbp"),
        col("active_subscriptions"),
        col("mrr_contribution_percent")
    ).orderBy(col("total_mrr_gbp").desc())

# Calculate cumulative contribution for top customers
window_spec = Window.orderBy(desc("total_mrr_gbp")).rowsBetween(Window.unboundedPreceding, Window.currentRow)
top_customers_cumulative = top_customers.withColumn(
    "cumulative_mrr",
    _sum("total_mrr_gbp").over(window_spec)
).withColumn(
    "cumulative_contribution_percent",
    spark_round((col("cumulative_mrr") / total_mrr) * 100, 2)
)

print("\n=== TOP 20 CUSTOMERS BY RECURRING REVENUE ===")
display(top_customers_cumulative.limit(20))

# ===== TOP PRODUCTS BY RECURRING REVENUE =====
top_products = active_subscriptions.groupBy("product_id").agg(
    _sum("MRR_GBP").alias("total_mrr_gbp"),
    _sum("Revenue_GBP").alias("total_revenue_gbp"),
    count("opportunity_id").alias("active_subscriptions"),
    countDistinct("customer_id").alias("unique_customers")
).withColumn(
    "mrr_contribution_percent",
    spark_round((col("total_mrr_gbp") / total_mrr) * 100, 2)
).join(dim_product, "product_id", "inner") \
    .select(
        col("product_id"),
        col("product_name"),
        col("plan_name"),
        col("billing_cycle"),
        spark_round(col("total_mrr_gbp"), 2).alias("total_mrr_gbp"),
        spark_round(col("total_revenue_gbp"), 2).alias("total_revenue_gbp"),
        col("active_subscriptions"),
        col("unique_customers"),
        col("mrr_contribution_percent")
    ).orderBy(col("total_mrr_gbp").desc())

print("\n=== TOP 20 PRODUCTS BY RECURRING REVENUE ===")
display(top_products.limit(20))

# ===== REVENUE CONCENTRATION ANALYSIS =====
# Calculate what percentage of revenue comes from top N customers/products
top_10_customers_mrr = top_customers.limit(10).agg(_sum("total_mrr_gbp")).collect()[0][0]
top_20_customers_mrr = top_customers.limit(20).agg(_sum("total_mrr_gbp")).collect()[0][0]

top_5_products_mrr = top_products.limit(5).agg(_sum("total_mrr_gbp")).collect()[0][0]
top_10_products_mrr = top_products.limit(10).agg(_sum("total_mrr_gbp")).collect()[0][0]

top_10_customers_pct = (top_10_customers_mrr / total_mrr * 100) if total_mrr > 0 else 0
top_20_customers_pct = (top_20_customers_mrr / total_mrr * 100) if total_mrr > 0 else 0
top_5_products_pct = (top_5_products_mrr / total_mrr * 100) if total_mrr > 0 else 0
top_10_products_pct = (top_10_products_mrr / total_mrr * 100) if total_mrr > 0 else 0

concentration_data = [
    ("Top 10 Customers", round(top_10_customers_mrr, 2), round(top_10_customers_pct, 2)),
    ("Top 10 Products", round(top_10_products_mrr, 2), round(top_10_products_pct, 2))
]

concentration_df = spark.createDataFrame(
    concentration_data,
    ["segment", "total_mrr_gbp", "contribution_percent"]
)

print("\n=== REVENUE CONCENTRATION ANALYSIS ===")
display(concentration_df)

print("\n=== Key Insights ===")
print(f"Total Active MRR: {total_mrr:,.2f}")
print(f"\nCustomer Concentration:")
print(f"  • Top 10 customers contribute: {top_10_customers_mrr:,.2f} ({top_10_customers_pct:.1f}%)")
print(f"\nProduct Concentration:")
print(f"  • Top 10 products contribute: {top_10_products_mrr:,.2f} ({top_10_products_pct:.1f}%)")

if top_10_customers_pct > 50:
    print(f"\n High customer concentration risk: Top 10 customers represent {top_10_customers_pct:.1f}% of MRR")
else:
    print(f"\n Healthy customer diversification: Top 10 customers represent {top_10_customers_pct:.1f}% of MRR")

In [0]:
from pyspark.sql.functions import year, month, col, sum as _sum, round as spark_round, date_format, to_date, concat, lit, avg
from pyspark.sql.window import Window

# Read fact table
fact_df = spark.table("gold_catalog.default.fact_subscription_table")

# Calculate monthly MRR for active subscriptions
monthly_mrr = fact_df.filter(col("close_status").isin(["ACTIVE", "WON"])) \
    .withColumn("year_month", date_format(col("start_date"), "yyyy-MM")) \
    .groupBy("year_month").agg(
        _sum("MRR_GBP").alias("monthly_mrr"),
        _sum("Revenue_GBP").alias("monthly_revenue")
    ).withColumn(
        "year_month_date", to_date(concat(col("year_month"), lit("-01")), "yyyy-MM-dd")
    ).orderBy("year_month_date")

# Calculate rolling 12-month metrics using window function
window_spec = Window.orderBy("year_month_date").rowsBetween(-11, 0)

rolling_12m_trend = monthly_mrr.withColumn(
    "rolling_12m_mrr",
    spark_round(_sum("monthly_mrr").over(window_spec), 2)
).withColumn(
    "rolling_12m_revenue",
    spark_round(_sum("monthly_revenue").over(window_spec), 2)
).withColumn(
    "rolling_12m_avg_mrr",
    spark_round(avg("monthly_mrr").over(window_spec), 2)
).select(
    col("year_month"),
    spark_round(col("monthly_mrr"), 2).alias("monthly_mrr"),
    spark_round(col("monthly_revenue"), 2).alias("monthly_revenue"),
    col("rolling_12m_mrr"),
    col("rolling_12m_revenue"),
    col("rolling_12m_avg_mrr")
).orderBy("year_month")

print("=== Rolling 12-Month Recurring Revenue Trend ===")
print("Note: First 11 months show partial rolling windows")
print("="*80)

display(rolling_12m_trend)

# Calculate growth metrics for complete 12-month periods
complete_periods = rolling_12m_trend.collect()

if len(complete_periods) >= 12:
    # Compare first complete 12-month period vs latest
    first_complete = complete_periods[11]  # First complete 12-month period
    latest = complete_periods[-1]  # Latest period
    
    growth = latest['rolling_12m_mrr'] - first_complete['rolling_12m_mrr']
    growth_pct = (growth / first_complete['rolling_12m_mrr'] * 100) if first_complete['rolling_12m_mrr'] > 0 else 0
    
    print(f"\n=== Key Insights ===")
    print(f"\nFirst Complete 12M Period ({first_complete['year_month']}):")
    print(f"  • Rolling 12M MRR: {first_complete['rolling_12m_mrr']:,.2f}")
    print(f"  • Rolling 12M Revenue: {first_complete['rolling_12m_revenue']:,.2f}")
    
    print(f"\nLatest 12M Period ({latest['year_month']}):")
    print(f"  • Rolling 12M MRR: {latest['rolling_12m_mrr']:,.2f}")
    print(f"  • Rolling 12M Revenue: {latest['rolling_12m_revenue']:,.2f}")
    print(f"  • Avg Monthly MRR: {latest['rolling_12m_avg_mrr']:,.2f}")
    
    print(f"\nGrowth from First to Latest Period:")
    print(f"  • Absolute Change: {growth:,.2f}")
    print(f"  • Percent Change: {growth_pct:+.2f}%")
    
    # Calculate month-over-month trend for recent periods
    if len(complete_periods) >= 13:
        recent_periods = complete_periods[-6:]  # Last 6 months
        print(f"\n=== Recent Trend (Last 6 Months) ===")
        for i in range(1, len(recent_periods)):
            prev = recent_periods[i-1]
            curr = recent_periods[i]
            mom_change = curr['rolling_12m_mrr'] - prev['rolling_12m_mrr']
            mom_pct = (mom_change / prev['rolling_12m_mrr'] * 100) if prev['rolling_12m_mrr'] > 0 else 0
            print(f"  • {curr['year_month']}: {curr['rolling_12m_mrr']:,.2f} ({mom_pct:+.2f}% MoM)")
    
    if growth_pct > 0:
        print(f"\n Rolling 12M MRR is UP {growth_pct:.2f}% overall")
    else:
        print(f"\n Rolling 12M MRR is DOWN {abs(growth_pct):.2f}% overall")
else:
    print(f"\n Only {len(complete_periods)} months of data available. Need 12+ months for complete analysis.")